# Clase 122 — PyTorch: tensores + autograd

Intentamos `import torch`. Si no está, fallback con numpy y autograd manual.

In [ ]:
USE_TORCH = False
try:
    import torch
    USE_TORCH = True
    print('torch:', torch.__version__, '| device cpu')
except Exception as e:
    print('torch no disponible. Fallback numpy + autograd manual. Motivo:', type(e).__name__)
import numpy as np
np.random.seed(42)

## 1. Tensores: creación + operaciones + broadcasting

In [ ]:
if USE_TORCH:
    torch.manual_seed(42)
    a = torch.tensor([[1., 2.], [3., 4.]])
    b = torch.ones(2, 2)
    print('a + b =\n', a + b)
    print('a @ b =\n', a @ b)
    print('broadcast a + [10, 20] =\n', a + torch.tensor([10., 20.]))
    print('device:', a.device)
else:
    a = np.array([[1., 2.], [3., 4.]])
    b = np.ones((2, 2))
    print('a + b =\n', a + b)
    print('a @ b =\n', a @ b)

## 2. Autograd: requires_grad + backward()

In [ ]:
if USE_TORCH:
    x = torch.tensor(3.0, requires_grad=True)
    y = x**2 + 2*x + 1   # (x+1)^2; dy/dx = 2x+2 = 8 en x=3
    y.backward()
    print(f'x={x.item()}, y={y.item()}, dy/dx={x.grad.item()} (esperado 8)')
else:
    # autograd manual con regla de la cadena
    x_val = 3.0
    y_val = x_val**2 + 2*x_val + 1
    dydx = 2*x_val + 2
    print(f'x={x_val}, y={y_val}, dy/dx={dydx} (esperado 8)')

## 3. Gradiente acumulado (importante en RNNs / gradient accumulation)

In [ ]:
if USE_TORCH:
    x = torch.tensor(1.0, requires_grad=True)
    for step in range(3):
        y = x**2
        y.backward()
        print(f'step {step}: x.grad acumulado = {x.grad.item()}')
    x.grad.zero_()
    print('después de zero_grad:', x.grad.item())
else:
    print('numpy: simular acumulación manualmente sumando dy/dx step a step')

## 4. nn.Module mínimo: regresión lineal manual

In [ ]:
# Dataset sintético: y = 3x + 2 + noise
X_np = np.linspace(-2, 2, 100).reshape(-1, 1).astype(np.float32)
y_np = (3*X_np + 2 + np.random.normal(0, 0.3, X_np.shape)).astype(np.float32)

if USE_TORCH:
    import torch.nn as nn
    X = torch.from_numpy(X_np); y = torch.from_numpy(y_np)
    model = nn.Linear(1, 1)
    opt = torch.optim.SGD(model.parameters(), lr=0.05)
    loss_fn = nn.MSELoss()
    for epoch in range(200):
        pred = model(X); loss = loss_fn(pred, y)
        opt.zero_grad(); loss.backward(); opt.step()
    w, b = model.weight.item(), model.bias.item()
    print(f'aprendido: w={w:.3f} (esp 3), b={b:.3f} (esp 2)')
else:
    # SGD manual
    w, b = 0.0, 0.0; lr = 0.05
    for epoch in range(200):
        pred = w*X_np + b
        err = pred - y_np
        dw = (2 * err * X_np).mean(); db = (2 * err).mean()
        w -= lr*dw; b -= lr*db
    print(f'aprendido: w={w:.3f} (esp 3), b={b:.3f} (esp 2)')

## 5. Derivada analítica vs autograd

In [ ]:
if USE_TORCH:
    x = torch.linspace(-3, 3, 7, requires_grad=True)
    y = (x**3).sum()   # dy/dx_i = 3 x_i^2
    y.backward()
    print('autograd:', x.grad.numpy())
    print('analytical:', 3 * (x.detach().numpy()**2))
else:
    xs = np.linspace(-3, 3, 7)
    print('analytical 3x^2:', 3 * xs**2)

## Conclusiones

- Tensores PyTorch = numpy + GPU + autograd.
- `.backward()` recorre el grafo dinámico que se construye durante el forward.
- Gradientes se *acumulan* — siempre `zero_grad()` al inicio de cada step.
- `nn.Module` + `optim` ahorran código pero el ciclo manual es importante para custom loops (Clase 121).

## ✅ Soluciones de los ejercicios

Fundamentos de PyTorch (tensores y autograd). Se validan por AST sin `torch`. El **Ej. 2** (autograd, dy/dx=12) lleva `assert` para que quede autoverificado cuando torch esté disponible. Cubre tensores/GPU, autograd, un MLP `nn.Module`, un training loop manual y `DataLoader`.

**Ej. 1 — Tensores.** Crear, mover a GPU y operar; puente con NumPy.

In [ ]:
import torch
import numpy as np

t = torch.tensor([[1., 2.], [3., 4.]])
print("shape:", t.shape, "| dtype:", t.dtype, "| device:", t.device)
if torch.cuda.is_available():
    t = t.to("cuda")
print("suma:", (t + t).cpu().numpy())
print("numpy -> torch (comparten memoria en CPU):", torch.from_numpy(np.ones(3)))

**Ej. 2 — Autograd.** `y = x^3`, `y.backward()` -> `x.grad = 3x^2 = 12` en `x=2`.

In [ ]:
import torch

x = torch.tensor([2.0], requires_grad=True)
y = x ** 3
y.backward()                       # calcula dy/dx y lo acumula en x.grad
print("dy/dx en x=2:", x.grad.item())   # 3 * 2^2 = 12
assert abs(x.grad.item() - 12.0) < 1e-6

**Ej. 3 — MLP custom.** Clase `nn.Module` con 2 `Linear` + ReLU; contar parámetros.

In [ ]:
import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

model = MLP()
print("n params:", sum(p.numel() for p in model.parameters()))   # 784*128+128 + 128*10+10

**Ej. 4 — Training loop manual.** 1 época en Fashion-MNIST, reportar loss promedio.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X = torch.randn(60000, 784); Y = torch.randint(0, 10, (60000,))
dl = DataLoader(TensorDataset(X, Y), batch_size=64, shuffle=True)

model = MLP()                      # del Ej. 3
opt = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.CrossEntropyLoss()

model.train()
total = 0.0
for xb, yb in dl:
    opt.zero_grad()
    loss = loss_fn(model(xb), yb)
    loss.backward()
    opt.step()
    total += loss.item()
print("loss promedio de la epoca:", total / len(dl))

**Ej. 5 — DataLoader.** `batch_size=32, shuffle=True, num_workers=2`; iterar un batch.

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X = torch.randn(1000, 784); Y = torch.randint(0, 10, (1000,))
dl = DataLoader(TensorDataset(X, Y), batch_size=32, shuffle=True, num_workers=2)
xb, yb = next(iter(dl))
print("batch:", xb.shape, yb.shape)   # (32, 784) (32,)
# nota Windows: num_workers>0 requiere el bloque `if __name__ == "__main__":` en scripts .py